# 02 — Transformação e Construção da Feature Matrix

**Pré-requisito:** notebook `01_extracao.ipynb` já executado (dados brutos em `data/raw/`).

**Objetivo:** transformar os dados brutos extraídos em um único dataset analítico pronto para a regressão. Tudo que é *transformação* mora aqui — limpeza, classificação LBCE, agregação, joins, feature engineering.

**Estrutura temporal:** painel de **3 anos (2022, 2023, 2024)** × ~5.570 municípios. Cada linha de `feature_matrix.parquet` é uma observação município × ano. As séries temporais permitem:

- **Suavizar ruído de amostragem** — taxas brutas em municípios pequenos são voláteis ano a ano.
- **Controlar choques de período** — efeitos comuns a todos os municípios em um dado ano (pós-COVID, mudanças no SUS).
- **Capturar tendência local** — variação intra-município entre 2022→2024 (despesa em saúde, óbitos).

> ⚠️ **Atenção sobre as fontes:** Atlas DH é **invariante no tempo** (Censo 2010 — IDH-M, Gini, RDPC etc. são repetidos em todas as linhas). População 2023 é **proxy do Censo 2022** (IBGE não publicou estimativa); PIB 2024 é **proxy do PIB 2023** (lag estrutural). Detalhe na coluna `ANO_FONTE` dos arquivos brutos.

## Etapas

1. **LBCE inline** — listas de CID-10 dos 5 grupos da Lista Brasileira de Causas Evitáveis e função de classificação.
2. **SIM** — carregar parquets brutos (81 arquivos: 27 UFs × 3 anos), limpar tipos/datas, decodificar `IDADE`, aplicar filtro 5–74 anos + classificação LBCE.
3. **Agregação** — contagem de óbitos evitáveis por município × ano × grupo.
4. **Join socioeconômico** — população (IBGE), PIB (IBGE), despesa em saúde (SICONFI), IDH-M/Gini (Atlas Brasil).
5. **Feature engineering** — calcular taxas per capita, log-transforms, encoding de UF/região.
6. **Persistência** — `feature_matrix.parquet` (entrada do notebook 03 de regressão).

## Saída

```
data/processed/
├── sim_evitaveis.parquet           # SIM filtrado e classificado (microdado)
├── obitos_municipio.parquet        # contagem agregada por município/ano (painel)
├── obitos_municipio_grupo.parquet  # contagem por município/ano/grupo LBCE
└── feature_matrix.parquet          # painel município/ano para ML
```

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT          = Path.cwd().parent
RAW_DIR       = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SIM_DIR     = RAW_DIR / "sim"
IBGE_DIR    = RAW_DIR / "ibge"
SICONFI_DIR = RAW_DIR / "siconfi"
ATLAS_DIR   = RAW_DIR / "atlas"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("Imports OK")

Imports OK


## 1. Lista Brasileira de Causas Evitáveis (LBCE) — 5 a 74 anos

**Referência oficial:** DATASUS/SVS-MS — *Lista de Tabulação de Causas Evitáveis de 5 a 74 anos*, baseada em:
- Malta DC et al. *Tabela Brasileira de Causas de Mortes Evitáveis*, Epidemiol. Serv. Saúde 16(4):233-244, 2007.
- Malta DC et al. *Atualização da lista*, Epidemiol. Serv. Saúde 19(2):173-176, 2010.

A lista divide as causas evitáveis em **5 grupos** segundo o tipo de intervenção do SUS que reduziria a morte:

### 1.1 — Reduzíveis por ações de imunoprevenção
Mortes que poderiam ser evitadas por vacinação.

In [2]:
IMUNOPREVENCAO = [
    "A17",   # Tuberculose do sistema nervoso (BCG)
    "A19",   # Tuberculose miliar (BCG)
    "A34",   # Tétano obstétrico
    "A35",   # Tétano
    "A36",   # Difteria
    "A37",   # Coqueluche
    "A80",   # Poliomielite aguda
    "B05",   # Sarampo
    "B06",   # Rubéola
    "B16",   # Hepatite aguda B
    "G000",  # Meningite por Haemophilus (G00.0)
]

### 1.2 — Reduzíveis por promoção, prevenção e atenção: doenças infecciosas
Tuberculose (diagnóstico/tratamento), HIV, hepatites, DSTs, infecções respiratórias, infecções da pele, doenças tropicais, etc.

In [3]:
DOENCAS_INFECCIOSAS = [
    # Tuberculose (diagnóstico e tratamento)
    "A15", "A16", "A18", "B90",

    # Doenças infecciosas intestinais (A00–A09)
    "A00", "A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08", "A09",

    # HIV/AIDS (B20–B24)
    "B20", "B21", "B22", "B23", "B24",

    # Hepatites virais (exceto B16 — hepatite B aguda, que está em imunoprevenção)
    "B15", "B17", "B18", "B19",

    # Sífilis, gonorreia e outras DSTs (A50–A59, A63–A64)
    "A50", "A51", "A52", "A53", "A54", "A55", "A56", "A57", "A58", "A59",
    "A63", "A64",

    # Doenças inflamatórias dos órgãos pélvicos femininos (N70–N76, exceto N73.6)
    "N70", "N71", "N72", "N73", "N74", "N75", "N76",

    # Febre reumática aguda e doenças reumáticas crônicas do coração (I00–I09)
    "I00", "I01", "I02", "I03", "I04", "I05", "I06", "I07", "I08", "I09",

    # Infecções respiratórias / pneumonia / influenza
    "J00", "J01", "J02", "J03", "J04", "J05", "J06",
    "J10", "J11", "J12", "J13", "J14", "J15", "J16", "J17", "J18",
    "J19", "J20", "J21", "J22",

    # Infecções da pele e do tecido subcutâneo (L02–L08)
    "L02", "L03", "L04", "L05", "L06", "L07", "L08",

    # Infecção do trato urinário (N39.0)
    "N390",

    # Doenças de notificação compulsória
    "A20", "A21", "A22",     # Zoonoses bacterianas (peste, tularemia, antraz)
    "A27",                     # Leptospirose
    "A30",                     # Hanseníase
    "A77",                     # Febre maculosa (rickettsiose)
    "A82",                     # Raiva
    "A90", "A91",             # Dengue / dengue hemorrágico
    "A923",                    # A92.3 — febre hemorrágica do Nilo Ocidental
    "A95",                     # Febre amarela
    "A985",                    # A98.5 — febre hemorrágica com síndrome renal
    "B03",                     # Varíola
    "B55",                     # Leishmaniose
    "B570", "B571", "B572",   # B57.0–B57.2 — Doença de Chagas
    "B65",                     # Esquistossomose

    # Outras infecções
    "A23", "A24", "A25", "A26",
    "A28", "A31", "A32", "A38",
    "A39", "A40", "A41",
    "A46",
    "A692",                    # A69.2 — Doença de Lyme
    "B50", "B51", "B52", "B53", "B54",   # Paludismo (malária)
    "G001", "G002", "G003", "G004", "G005", "G006", "G007", "G008", "G009",  # Meningites bacterianas
    "G01",
]

### 1.3 — Reduzíveis por promoção, prevenção e atenção: doenças não transmissíveis
Neoplasias com rastreio/tratamento, doenças cardiovasculares, diabetes, doenças respiratórias crônicas, transtornos do álcool, etc.

In [4]:
DOENCAS_NAO_TRANSMISSIVEIS = [
    # Neoplasias malignas
    "C00",                                # Lábio
    "C43", "C44",                         # Melanoma e outras neoplasias da pele
    "C22",                                # Fígado e vias biliares intra-hepáticas
    "C16",                                # Estômago
    "C18", "C19", "C20", "C21",          # Cólon, retossigmoide, reto e ânus
    "C01", "C02", "C03", "C04", "C05", "C06",  # Boca
    "C09", "C10",                         # Orofaringe
    "C12", "C13", "C14",                  # Hipofaringe
    "C32",                                # Laringe
    "C15",                                # Esôfago
    "C33", "C34",                         # Traqueia, brônquios e pulmão
    "C50",                                # Mama
    "C53",                                # Colo do útero
    "C62",                                # Testículos
    "C73",                                # Glândula tireoide
    "C81",                                # Doença de Hodgkin
    "C91",                                # Leucemia linfoide
    "C92",                                # Leucemia mieloide

    # Doenças endócrinas e metabólicas
    "E01", "E02", "E03", "E04", "E05",   # Tireotoxicose, hipotireoidismo, deficiência de iodo
    "E10", "E11", "E12", "E13", "E14",   # Diabetes mellitus
    "E66",                                 # Obesidade

    # Doenças relacionadas ao álcool
    "F10",                                  # Transtornos mentais pelo álcool
    "I426",                                 # I42.6 — Cardiomiopatia alcoólica
    "K292",                                 # K29.2 — Gastrite alcoólica
    "K70",                                  # Doença hepática alcoólica
    "K860",                                 # K86.0 — Pancreatite crônica alcoólica

    # Sistema nervoso
    "G40", "G41",                          # Epilepsia e estado de mal epiléptico

    # Aparelho circulatório
    "I10", "I11", "I12", "I13",            # Hipertensivas (exceto I15 = secundária)
    "I20", "I21", "I22", "I23", "I24", "I25",  # Doenças isquêmicas do coração
    "I50",                                  # Insuficiência cardíaca
    "I70",                                  # Aterosclerose
    "I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68", "I69",  # Cerebrovasculares

    # Aparelho respiratório
    "J40", "J41", "J42", "J43", "J44", "J45", "J46", "J47",  # Doenças crônicas vias aéreas
    "J81",                                                       # Edema pulmonar
    "J60", "J61", "J62", "J63", "J64", "J65", "J66", "J67", "J68", "J69", "J70",  # Agentes externos

    # Aparelho digestivo
    "K25", "K26", "K27", "K28",            # Úlceras
    "K35",                                  # Apendicite aguda
    "K40", "K41", "K42", "K43", "K44", "K45", "K46",  # Hérnias
    "K56",                                  # Íleo paralítico e obstrução intestinal
    "K80", "K81", "K82", "K83",            # Vesícula biliar e vias biliares

    # Aparelho geniturinário
    "N18",                                  # Insuficiência renal crônica
]

### 1.4 — Reduzíveis por prevenção e atenção às causas de morte materna
Gravidez, parto e puerpério (CID-10 capítulo O).

In [5]:
# O00–O26, O29–O99 (O27 não existe; O28 — achados anormais exame antenatal — é excluído)
MORTE_MATERNA = (
    [f"O{i:02d}" for i in range(0, 27)] +
    [f"O{i:02d}" for i in range(29, 100)]
)

### 1.5 — Reduzíveis por ações intersetoriais (causas externas) — **EXCLUÍDAS da análise**
Acidentes, agressões, suicídios, intoxicações, eventos de intenção indeterminada.

> **Decisão analítica:** mantemos a lista definida abaixo por completude conceitual (a LBCE oficial inclui este grupo), mas **não usamos `causas_externas` no dataset de modelagem**. As variáveis socioeconômicas que temos disponíveis (PIB per capita, despesa em saúde, IDH-M, Gini, renda) explicam mal a mortalidade por causas externas — quedas, acidentes de trânsito e homicídios respondem a fatores como segurança pública, infraestrutura viária, álcool/drogas e violência interpessoal, que não estão no nosso conjunto de features. Incluir esse grupo introduziria ruído estrutural na regressão sem ganho explicativo. O escopo passa a ser **causas evitáveis sensíveis ao SUS e ao desenvolvimento socioeconômico**: imunoprevenção, doenças infecciosas, doenças crônicas não transmissíveis e morte materna.

In [6]:
CAUSAS_EXTERNAS = [
    # Acidentes de transporte (V01–V99)
    *[f"V{i:02d}" for i in range(1, 100)],

    # Quedas (W00–W19)
    *[f"W{i:02d}" for i in range(0, 20)],
    # Forças mecânicas inanimadas (W20–W49)
    *[f"W{i:02d}" for i in range(20, 50)],
    # Forças mecânicas animadas (W50–W64)
    *[f"W{i:02d}" for i in range(50, 65)],
    # Afogamento e submersão (W65–W74)
    *[f"W{i:02d}" for i in range(65, 75)],
    # Outros riscos acidentais à respiração (W75–W84)
    *[f"W{i:02d}" for i in range(75, 85)],
    # Corrente elétrica, radiação, temperaturas extremas (W85–W99)
    *[f"W{i:02d}" for i in range(85, 100)],

    # Fogo e chamas (X00–X09)
    *[f"X{i:02d}" for i in range(0, 10)],
    # Fonte de calor e substâncias quentes (X10–X19)
    *[f"X{i:02d}" for i in range(10, 20)],
    # Animais e plantas venenosas (X20–X29)
    *[f"X{i:02d}" for i in range(20, 30)],
    # Forças da natureza (X30–X39)
    *[f"X{i:02d}" for i in range(30, 40)],
    # Envenenamento/intoxicação acidental (X40–X49)
    *[f"X{i:02d}" for i in range(40, 50)],
    # Exposição acidental a outros fatores (X58–X59)
    "X58", "X59",
    # Lesões autoprovocadas / suicídio (X60–X84)
    *[f"X{i:02d}" for i in range(60, 85)],
    # Agressões (X85–Y09)
    *[f"X{i:02d}" for i in range(85, 100)],
    *[f"Y{i:02d}" for i in range(0, 10)],

    # Eventos de intenção indeterminada (Y10–Y34)
    *[f"Y{i:02d}" for i in range(10, 35)],
    # Intervenções legais e operações de guerra (Y35–Y36)
    "Y35", "Y36",
    # Efeitos adversos de drogas/medicamentos (Y40–Y59)
    *[f"Y{i:02d}" for i in range(40, 60)],
    # Acidentes em pacientes durante cuidados médicos (Y60–Y69)
    *[f"Y{i:02d}" for i in range(60, 70)],
    # Incidentes adversos durante atos diagnósticos com dispositivos (Y70–Y82)
    *[f"Y{i:02d}" for i in range(70, 83)],
    # Reação anormal / complicação tardia de procedimentos (Y83–Y84)
    "Y83", "Y84",
]

### 1.6 — Função de classificação
Constrói um índice prefixo→grupo (lookup O(1)) e expõe `classificar_causa(cid)` que retorna o nome do grupo ou `None`.

In [7]:
# GRUPOS_LBCE = grupos usados efetivamente na análise.
# `causas_externas` está definida acima mas é deliberadamente excluída
# (ver justificativa na seção 1.5).
GRUPOS_LBCE = {
    "imunoprevencao":             IMUNOPREVENCAO,
    "doencas_infecciosas":        DOENCAS_INFECCIOSAS,
    "doencas_nao_transmissiveis": DOENCAS_NAO_TRANSMISSIVEIS,
    "morte_materna":              MORTE_MATERNA,
}

_INDICE_LBCE = {
    codigo.replace(".", "").strip().upper(): grupo
    for grupo, codigos in GRUPOS_LBCE.items()
    for codigo in codigos
}


def classificar_causa(causa_basica) -> str | None:
    """
    Classifica um código CID-10 em grupo LBCE (apenas grupos sensíveis a SUS/IDH).
    Tenta match com 4 chars (subcategoria) e depois 3 chars (categoria).
    Retorna nome do grupo ou None.
    """
    if not isinstance(causa_basica, str) or not causa_basica:
        return None
    causa = causa_basica.strip().upper().replace(".", "")
    if not causa:
        return None
    for n in (4, 3):
        if causa[:n] in _INDICE_LBCE:
            return _INDICE_LBCE[causa[:n]]
    return None


# Sanity check
print(f"Total de prefixos CID-10 indexados: {len(_INDICE_LBCE):,}")
print(f"Por grupo:")
for g, codigos in GRUPOS_LBCE.items():
    print(f"  {g:<32}: {len(codigos):>4}")

# V50 (acidente de trânsito) agora retorna None — causas externas foram excluídas.
for cid, esperado in [("I21", "doencas_nao_transmissiveis"), ("A17", "imunoprevencao"),
                       ("O05", "morte_materna"), ("V50", None), ("Z99", None)]:
    assert classificar_causa(cid) == esperado, f"{cid} → {classificar_causa(cid)} (esperado: {esperado})"
print("\nTestes OK")

Total de prefixos CID-10 indexados: 344
Por grupo:
  imunoprevencao                  :   11
  doencas_infecciosas             :  126
  doencas_nao_transmissiveis      :  109
  morte_materna                   :   98

Testes OK


## 2. Carregar SIM bruto

Lê todos os parquets gerados pelo notebook 01 (um por UF/ano) e concatena.

In [8]:
parquet_paths = sorted(SIM_DIR.glob("DO*.parquet"))
print(f"Parquets encontrados: {len(parquet_paths)}")

frames = []
for p in parquet_paths:
    df_p = pd.read_parquet(p)
    df_p["_arquivo"] = p.name
    frames.append(df_p)

df_sim_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal bruto: {len(df_sim_raw):,} registros, {df_sim_raw.shape[1]} colunas")

Parquets encontrados: 81



Total bruto: 4,541,891 registros, 88 colunas


## 3. Limpeza de tipos, datas, decodificação de IDADE

- Datas SIM são strings `DDMMAAAA` → converter para `datetime`.
- `IDADE` no SIM é codificada `XYY` (`X` = unidade): `4XX` = anos, `5XX` = anos + 100, `3XX` = meses, etc.

In [9]:
df = df_sim_raw.copy()

# Datas — formato DDMMAAAA
for col in ("DTOBITO", "DTNASC"):
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col].astype(str).str.strip().str.zfill(8),
            format="%d%m%Y",
            errors="coerce",
        )

# Strings — strip e "" → NA
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Categóricas
for col in ("SEXO", "RACACOR", "ESTCIV", "ESC"):
    if col in df.columns:
        df[col] = df[col].astype("category")

# CODMUNRES sempre 6 dígitos (zero à esquerda)
if "CODMUNRES" in df.columns:
    df["CODMUNRES"] = df["CODMUNRES"].astype(str).str.zfill(6)

print("Limpeza de tipos OK")

Limpeza de tipos OK


In [10]:
def decode_idade(valor) -> float:
    """Decodifica IDADE do SIM (XYY) para anos (float). NaN se inválido."""
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return np.nan
    s = str(valor).strip().zfill(3)
    if len(s) != 3 or not s.isdigit():
        return np.nan
    unidade, vv = s[0], int(s[1:])
    if unidade in ("0", "1"):
        return vv / (24 * 365.25)         # horas → anos
    if unidade == "2":
        return vv / 365.25                # dias → anos
    if unidade == "3":
        return vv / 12                    # meses → anos
    if unidade == "4":
        return float(vv)                  # 0–99 anos
    if unidade == "5":
        return float(vv + 100)            # 100+ anos
    return np.nan                          # 9 = ignorado


df["IDADE_ANOS"] = df["IDADE"].map(decode_idade)
df["ANO_OBITO"]  = df["DTOBITO"].dt.year.astype("Int64")

print("Distribuição da idade:")
print(df["IDADE_ANOS"].describe().round(1))
print(f"\nIdade ignorada / inválida: {df['IDADE_ANOS'].isna().sum():,}")

Distribuição da idade:


count    4535868.0
mean          66.8
std           21.5
min            0.0
25%           57.0
50%           71.0
75%           82.0
max          131.0
Name: IDADE_ANOS, dtype: float64

Idade ignorada / inválida: 6,023


## 4. Aplicar LBCE — filtro 5–74 anos + classificação

Mantemos `df_5_74` com a flag `EVITAVEL` para análises de denominador (todos os óbitos da faixa) e `df_evitaveis` apenas com causas evitáveis.

In [11]:
df_5_74 = df[(df["IDADE_ANOS"] >= 5) & (df["IDADE_ANOS"] < 75)].copy()
df_5_74["GRUPO_LBCE"] = df_5_74["CAUSABAS"].map(classificar_causa)
df_5_74["EVITAVEL"]   = df_5_74["GRUPO_LBCE"].notna()

df_evitaveis = df_5_74[df_5_74["EVITAVEL"]].copy()

print(f"Total bruto                       : {len(df):>12,}")
print(f"Faixa 5–74 anos                   : {len(df_5_74):>12,}")
print(f"Óbitos evitáveis (LBCE)           : {len(df_evitaveis):>12,}")
print(f"\nDistribuição por grupo LBCE:")
print(df_evitaveis["GRUPO_LBCE"].value_counts())

Total bruto                       :    4,541,891
Faixa 5–74 anos                   :    2,486,070
Óbitos evitáveis (LBCE)           :    1,366,467

Distribuição por grupo LBCE:
GRUPO_LBCE
doencas_nao_transmissiveis    1109236
doencas_infecciosas            250735
morte_materna                    4827
imunoprevencao                   1669
Name: count, dtype: int64


In [12]:
# Persiste o SIM filtrado
out_sim = PROCESSED_DIR / "sim_evitaveis.parquet"
df_evitaveis.to_parquet(out_sim, index=False)
print(f"Salvo: {out_sim.name} ({out_sim.stat().st_size / 1e6:.1f} MB)")

Salvo: sim_evitaveis.parquet (70.1 MB)


## 5. Agregação de óbitos por município × ano × grupo

Reduz o microdado a um nível pronto para a regressão. Duas visões:

- **`obitos_municipio`** — total de óbitos evitáveis por município/ano (1 linha por município/ano).
- **`obitos_municipio_grupo`** — desdobrado por grupo LBCE (5 linhas por município/ano, no máximo).

In [13]:
# Total por município/ano
obitos_municipio = (
    df_evitaveis
    .groupby(["CODMUNRES", "ANO_OBITO"], observed=True)
    .size()
    .reset_index(name="OBITOS_EVITAVEIS")
    .rename(columns={"CODMUNRES": "CODMUN6", "ANO_OBITO": "ANO"})
)
obitos_municipio["ANO"] = obitos_municipio["ANO"].astype(int)

# Por grupo LBCE
obitos_municipio_grupo = (
    df_evitaveis
    .groupby(["CODMUNRES", "ANO_OBITO", "GRUPO_LBCE"], observed=True)
    .size()
    .reset_index(name="OBITOS")
    .rename(columns={"CODMUNRES": "CODMUN6", "ANO_OBITO": "ANO"})
)
obitos_municipio_grupo["ANO"] = obitos_municipio_grupo["ANO"].astype(int)

print(f"obitos_municipio       : {len(obitos_municipio):,} linhas (município × ano)")
print(f"obitos_municipio_grupo : {len(obitos_municipio_grupo):,} linhas (município × ano × grupo)")
obitos_municipio.head()

obitos_municipio       : 16,733 linhas (município × ano)
obitos_municipio_grupo : 35,272 linhas (município × ano × grupo)


,CODMUN6,ANO,OBITOS_EVITAVEIS
0,110000,2023,1
1,110001,2022,55
2,110001,2023,56
3,110001,2024,39
4,110002,2022,209


## 6. Carregar features socioeconômicas

Cada source foi extraído pelo notebook 01:

In [14]:
# 6.1 — População IBGE
pop_files = sorted(IBGE_DIR.glob("populacao_municipal_*.csv"))
df_pop = pd.concat(
    [pd.read_csv(p, dtype={"CODMUN7": str, "CODMUN6": str}) for p in pop_files],
    ignore_index=True,
)
print(f"População : {len(df_pop):,} linhas | {df_pop['CODMUN6'].nunique():,} municípios")

População : 16,711 linhas | 5,571 municípios


In [15]:
# 6.2 — PIB IBGE
pib_files = sorted(IBGE_DIR.glob("pib_municipal_*.csv"))
df_pib = pd.concat(
    [pd.read_csv(p, dtype={"CODMUN7": str, "CODMUN6": str}) for p in pib_files],
    ignore_index=True,
)
print(f"PIB       : {len(df_pib):,} linhas | total Brasil: R$ {df_pib['PIB_RS_MIL'].sum()/1e6:,.1f} bi")

PIB       : 16,710 linhas | total Brasil: R$ 31,966.4 bi


In [16]:
# 6.3 — Despesa em saúde SICONFI (Despesas Empenhadas, função 10 - Saúde)
siconfi_files = sorted(SICONFI_DIR.glob("despesa_saude_*.csv"))
df_saude_raw = pd.concat([pd.read_csv(p) for p in siconfi_files], ignore_index=True)

# Mantém apenas "Despesas Empenhadas" (1 valor por município/ano)
df_saude = (
    df_saude_raw[df_saude_raw["COLUNA"] == "Despesas Empenhadas"]
    .copy()
    .rename(columns={"VALOR": "DESPESA_SAUDE_RS", "COD_IBGE": "CODMUN7"})
)
df_saude["CODMUN7"]          = df_saude["CODMUN7"].astype(str).str.strip()
df_saude["CODMUN6"]          = df_saude["CODMUN7"].str[:6]
df_saude["DESPESA_SAUDE_RS"] = pd.to_numeric(df_saude["DESPESA_SAUDE_RS"], errors="coerce")
df_saude = df_saude[["CODMUN6", "ANO", "DESPESA_SAUDE_RS"]].drop_duplicates(["CODMUN6", "ANO"])

print(f"Despesa Saúde: {len(df_saude):,} linhas | {df_saude['CODMUN6'].nunique():,} municípios")

Despesa Saúde: 16,632 linhas | 5,567 municípios


In [17]:
# 6.4 — Atlas DH (IPEA Data) — IDH-M, Gini, renda por quinto, etc.
atlas_files = sorted(ATLAS_DIR.glob("atlas_dh_ipea_*.csv"))
if atlas_files:
    df_atlas = pd.concat(
        [pd.read_csv(p, dtype={"CODMUN7": str, "CODMUN6": str}) for p in atlas_files],
        ignore_index=True,
    ).drop_duplicates("CODMUN6")
    indicadores = [c for c in df_atlas.columns if c not in ("CODMUN7", "CODMUN6")]
    print(f"Atlas DH (IPEA): {len(df_atlas):,} municípios | {len(indicadores)} indicadores")
    print(f"Indicadores: {indicadores}")
else:
    print("⚠️  Atlas DH não baixado. Rode a Seção 6 do notebook 01_extracao.")
    df_atlas = None

Atlas DH (IPEA): 5,564 municípios | 17 indicadores


Indicadores: ['IDHM', 'IDHM_E', 'IDHM_L', 'IDHM_R', 'GINI', 'RDPC', 'RDPC1', 'RDPC2', 'RDPC3', 'RDPC4', 'RDPC5', 'PMPOB', 'PIND', 'THEIL', 'ESPVIDA', 'T_ANALF15M', 'T_AGUA']


## 7. Construção da feature matrix

Junta tudo em uma única tabela com **1 linha por município × ano**, contendo:

- `OBITOS_EVITAVEIS` — outcome (target da regressão)
- `POPULACAO`, `PIB_RS_MIL`, `DESPESA_SAUDE_RS` — denominadores e features brutos
- `PIB_PER_CAPITA`, `DESPESA_SAUDE_PC`, `TAXA_EVITAVEL_100K` — derivados
- `IDHM`, `GINI`, `RDPC`, etc. — opcionais (Atlas)
- `UF`, `REGIAO` — categóricas geográficas

In [18]:
REGIAO_POR_UF = {
    "AC":"N","AM":"N","AP":"N","PA":"N","RO":"N","RR":"N","TO":"N",
    "AL":"NE","BA":"NE","CE":"NE","MA":"NE","PB":"NE","PE":"NE","PI":"NE","RN":"NE","SE":"NE",
    "DF":"CO","GO":"CO","MS":"CO","MT":"CO",
    "ES":"SE","MG":"SE","RJ":"SE","SP":"SE",
    "PR":"S","RS":"S","SC":"S",
}

def codmun_para_uf(cod6: str) -> str:
    """Os 2 primeiros dígitos do CODMUN6 são o código IBGE da UF."""
    UFS_BR = {
        11:"RO",12:"AC",13:"AM",14:"RR",15:"PA",16:"AP",17:"TO",
        21:"MA",22:"PI",23:"CE",24:"RN",25:"PB",26:"PE",27:"AL",28:"SE",29:"BA",
        31:"MG",32:"ES",33:"RJ",35:"SP",
        41:"PR",42:"SC",43:"RS",
        50:"MS",51:"MT",52:"GO",53:"DF",
    }
    return UFS_BR.get(int(cod6[:2]), "")

In [19]:
# Base = produto cartesiano de municípios IBGE × anos (garante linhas mesmo p/ municípios sem óbito)
anos = sorted(obitos_municipio["ANO"].unique().tolist())
base = (
    df_pop[["CODMUN6", "MUNICIPIO_NOME", "ANO", "POPULACAO"]]
    .copy()
    .query("ANO in @anos")
)

# Merge óbitos
fm = base.merge(obitos_municipio, on=["CODMUN6", "ANO"], how="left")
fm["OBITOS_EVITAVEIS"] = fm["OBITOS_EVITAVEIS"].fillna(0).astype(int)

# Merge PIB
fm = fm.merge(df_pib[["CODMUN6", "ANO", "PIB_RS_MIL"]], on=["CODMUN6", "ANO"], how="left")

# Merge Despesa em Saúde
fm = fm.merge(df_saude, on=["CODMUN6", "ANO"], how="left")

# Merge Atlas (se disponível) — Atlas é estático (Censo 2010), repete-se para todos os anos
if df_atlas is not None:
    fm = fm.merge(df_atlas, on="CODMUN6", how="left")

# Geografia
fm["UF"]     = fm["CODMUN6"].map(codmun_para_uf)
fm["REGIAO"] = fm["UF"].map(REGIAO_POR_UF)

print(f"Feature matrix: {fm.shape[0]:,} linhas × {fm.shape[1]} colunas")
fm.head()

Feature matrix: 16,711 linhas × 27 colunas


,CODMUN6,MUNICIPIO_NOME,ANO,POPULACAO,OBITOS_EVITAVEIS,PIB_RS_MIL,DESPESA_SAUDE_RS,CODMUN7,IDHM,IDHM_E,IDHM_L,IDHM_R,GINI,RDPC,RDPC1,RDPC2,RDPC3,RDPC4,RDPC5,PMPOB,PIND,THEIL,ESPVIDA,T_ANALF15M,T_AGUA,UF,REGIAO
0,110001,Alta Floresta D'Oeste - RO,2022,21494.0,55,919520.0,2.918174e+07,1100015,0.641,0.526,0.763,0.657,0.58,476.99,36.89,160.31,289.97,469.88,1424.61,26.04,14.29,0.60,70.75,11.99,93.69,RO,N
1,110002,Ariquemes - RO,2022,96833.0,209,3809355.0,1.193331e+08,1100023,0.702,0.600,0.806,0.716,0.53,689.95,120.08,270.77,420.44,638.10,1996.77,11.54,4.36,0.51,73.36,7.90,98.54,RO,N
2,110003,Cabixi - RO,2022,5351.0,8,289783.0,1.093209e+07,1100031,0.650,0.559,0.757,0.650,0.51,457.17,78.16,181.72,292.34,480.37,1256.80,21.20,7.27,0.44,70.39,13.63,95.49,RO,N
3,110004,Cacoal - RO,2022,86887.0,181,3195489.0,7.671583e+07,1100049,0.718,0.620,0.821,0.727,0.57,738.06,108.24,265.14,431.80,656.39,2247.76,13.08,5.97,0.61,74.27,8.29,97.96,RO,N
4,110005,Cerejeiras - RO,2022,15890.0,29,903099.0,2.560032e+07,1100056,0.692,0.602,0.799,0.688,0.50,577.18,104.82,236.32,389.57,581.95,1568.87,13.70,4.72,0.46,72.94,10.29,97.53,RO,N


In [20]:
# Features derivadas
fm["PIB_PER_CAPITA"]      = (fm["PIB_RS_MIL"] * 1_000) / fm["POPULACAO"]
fm["DESPESA_SAUDE_PC"]    = fm["DESPESA_SAUDE_RS"] / fm["POPULACAO"]
fm["TAXA_EVITAVEL_100K"]  = (fm["OBITOS_EVITAVEIS"] / fm["POPULACAO"]) * 100_000

# Log-transforms para variáveis com cauda longa (úteis em regressão linear)
for c in ("PIB_PER_CAPITA", "DESPESA_SAUDE_PC", "POPULACAO"):
    fm[f"LOG_{c}"] = np.log1p(fm[c])

# Sumário de qualidade do merge
print("Cobertura por feature (% não-nulos):")
for c in ("POPULACAO", "PIB_RS_MIL", "DESPESA_SAUDE_RS",
          "IDHM", "GINI", "RDPC"):
    if c in fm.columns:
        cobertura = fm[c].notna().mean() * 100
        print(f"  {c:<22}: {cobertura:>5.1f}%")

print(f"\nDescritiva da TAXA_EVITAVEL_100K:")
print(fm["TAXA_EVITAVEL_100K"].describe().round(1))

Cobertura por feature (% não-nulos):
  POPULACAO             : 100.0%
  PIB_RS_MIL            : 100.0%
  DESPESA_SAUDE_RS      :  99.5%
  IDHM                  :  99.9%
  GINI                  :  99.9%
  RDPC                  :  99.9%

Descritiva da TAXA_EVITAVEL_100K:
count    16710.0
mean       225.9
std         79.8
min          0.0
25%        173.1
50%        219.6
75%        270.5
max        865.8
Name: TAXA_EVITAVEL_100K, dtype: float64


### 7.1 Taxa padronizada por idade

A `TAXA_EVITAVEL_100K` é **bruta**: não corrige a estrutura etária do município. Como as causas evitáveis (5–74 anos) são dominadas por doenças crônicas que matam perto dos 70, municípios com população mais velha têm taxa bruta maior só por isso — gerando o paradoxo de Sul/Sudeste aparecerem acima de Norte/Nordeste.

Calculamos aqui a **taxa padronizada por idade** (padronização direta), com denominador = população por faixa quinquenal do Censo 2022 (extraída no notebook 01) e padrão = estrutura etária nacional. Para cada município × ano:

$$\text{taxa pad.} = \sum_{a} \frac{\text{óbitos}_a}{\text{pop}_a} \times w_a \times 100{.}000, \qquad w_a = \frac{\text{pop. nacional da faixa } a}{\text{pop. nacional 5--74}}$$

O denominador por idade é do Censo 2022 (mesmo para os 3 anos — a estrutura etária muda pouco). A coluna `TAXA_PADRONIZADA_100K` entra na feature matrix ao lado da bruta; a análise comparativa detalhada está no notebook 06.

In [21]:
# Faixas quinquenais 5-74 (mesmas do notebook 01)
FAIXAS_BINS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75]
FAIXAS_LBL  = ["05-09", "10-14", "15-19", "20-24", "25-29", "30-34", "35-39",
               "40-44", "45-49", "50-54", "55-59", "60-64", "65-69", "70-74"]

pop_idade_path = IBGE_DIR / "populacao_idade_municipal_2022.csv"
if pop_idade_path.exists():
    pop_idade = pd.read_csv(pop_idade_path, dtype={"CODMUN6": str})
    pop_idade["CODMUN6"] = pop_idade["CODMUN6"].str.zfill(6)

    # Óbitos evitáveis por município × ano × faixa etária
    ev = df_evitaveis[["CODMUNRES", "ANO_OBITO", "IDADE_ANOS"]].copy()
    ev["FAIXA"] = pd.cut(ev["IDADE_ANOS"], bins=FAIXAS_BINS, labels=FAIXAS_LBL, right=False)
    ev = ev[ev["FAIXA"].notna()]
    ob_faixa = (ev.groupby(["CODMUNRES", "ANO_OBITO", "FAIXA"], observed=True)
                  .size().reset_index(name="obitos")
                  .rename(columns={"CODMUNRES": "CODMUN6", "ANO_OBITO": "ANO"}))
    ob_faixa["ANO"] = ob_faixa["ANO"].astype(int)

    # Taxa específica por idade (denominador = pop Censo 2022, igual p/ os 3 anos)
    tx = ob_faixa.merge(pop_idade, on=["CODMUN6", "FAIXA"], how="left")
    tx = tx[(tx["POP"].notna()) & (tx["POP"] > 0)]
    tx["taxa_faixa"] = tx["obitos"] / tx["POP"]

    # Pesos da população-padrão nacional (estrutura etária Brasil 2022)
    padrao = pop_idade.groupby("FAIXA")["POP"].sum()
    peso   = (padrao / padrao.sum()).to_dict()
    tx["w"] = tx["FAIXA"].map(peso)

    # Taxa padronizada direta por município × ano (/100k)
    taxa_pad = (tx.assign(c=tx["taxa_faixa"] * tx["w"])
                  .groupby(["CODMUN6", "ANO"])["c"].sum()
                  .mul(100_000)
                  .rename("TAXA_PADRONIZADA_100K")
                  .reset_index())

    fm = fm.merge(taxa_pad, on=["CODMUN6", "ANO"], how="left")
    cob = fm["TAXA_PADRONIZADA_100K"].notna().mean() * 100
    print(f"TAXA_PADRONIZADA_100K adicionada | cobertura: {cob:.1f}%")
    print(fm.groupby("REGIAO")[["TAXA_EVITAVEL_100K", "TAXA_PADRONIZADA_100K"]]
            .mean().round(1).reindex(["N", "NE", "CO", "SE", "S"]))
else:
    print("⚠️  populacao_idade_municipal_2022.csv não encontrado — rode a Seção 3.1 do notebook 01.")
    fm["TAXA_PADRONIZADA_100K"] = np.nan

TAXA_PADRONIZADA_100K adicionada | cobertura: 99.9%
        TAXA_EVITAVEL_100K  TAXA_PADRONIZADA_100K
REGIAO                                           
N                    158.9                  227.6
NE                   204.1                  247.3
CO                   229.8                  257.4
SE                   244.6                  244.5
S                    256.5                  241.9


## 8. Persistência

Três parquets em `data/processed/`:

In [22]:
out_obitos = PROCESSED_DIR / "obitos_municipio.parquet"
out_grupo  = PROCESSED_DIR / "obitos_municipio_grupo.parquet"
out_fm     = PROCESSED_DIR / "feature_matrix.parquet"

obitos_municipio.to_parquet(out_obitos, index=False)
obitos_municipio_grupo.to_parquet(out_grupo, index=False)
fm.to_parquet(out_fm, index=False)

for p in (out_obitos, out_grupo, out_fm):
    print(f"  {p.name:<32} → {p.stat().st_size / 1e6:>6.2f} MB")

# Sanity check do painel: distribuição por ano
print("\nResumo do painel (1 linha = 1 município × 1 ano):")
resumo_painel = (
    fm.groupby("ANO")
      .agg(
          n_municipios   = ("CODMUN6", "nunique"),
          obitos_total   = ("OBITOS_EVITAVEIS", "sum"),
          taxa_media     = ("TAXA_EVITAVEL_100K", "mean"),
          taxa_mediana   = ("TAXA_EVITAVEL_100K", "median"),
          desp_saude_pc  = ("DESPESA_SAUDE_PC", "mean"),
      )
      .round(2)
)
print(resumo_painel)

  obitos_municipio.parquet         →   0.07 MB
  obitos_municipio_grupo.parquet   →   0.12 MB
  feature_matrix.parquet           →   2.21 MB

Resumo do painel (1 linha = 1 município × 1 ano):
      n_municipios  obitos_total  taxa_media  taxa_mediana  desp_saude_pc
ANO                                                                      
2022          5570        450311      226.95        219.61        1466.66
2023          5570        446605      222.50        216.83        1635.19
2024          5571        468760      228.33        221.98        1829.86


In [23]:
def carregar_feature_matrix(processed_dir: Path = None) -> pd.DataFrame:
    """Utilitário para o notebook 03: carrega a feature matrix persistida e
    garante tipos corretos após a desserialização do parquet."""
    if processed_dir is None:
        processed_dir = Path.cwd().parent / "data" / "processed"

    fm = pd.read_parquet(processed_dir / "feature_matrix.parquet")
    fm["CODMUN6"] = fm["CODMUN6"].astype(str).str.zfill(6)
    fm["ANO"]     = fm["ANO"].astype(int)
    return fm

In [24]:
# Sanity check: roundtrip do parquet
fm_check = carregar_feature_matrix()
print(f"{fm_check.shape[0]:,} linhas × {fm_check.shape[1]} colunas")
fm_check.head()

16,711 linhas × 34 colunas

,CODMUN6,MUNICIPIO_NOME,ANO,POPULACAO,OBITOS_EVITAVEIS,PIB_RS_MIL,DESPESA_SAUDE_RS,CODMUN7,IDHM,IDHM_E,IDHM_L,IDHM_R,GINI,RDPC,RDPC1,RDPC2,RDPC3,RDPC4,RDPC5,PMPOB,PIND,THEIL,ESPVIDA,T_ANALF15M,T_AGUA,UF,REGIAO,PIB_PER_CAPITA,DESPESA_SAUDE_PC,TAXA_EVITAVEL_100K,LOG_PIB_PER_CAPITA,LOG_DESPESA_SAUDE_PC,LOG_POPULACAO,TAXA_PADRONIZADA_100K
0,110001,Alta Floresta D'Oeste - RO,2022,21494.0,55,919520.0,2.918174e+07,1100015,0.641,0.526,0.763,0.657,0.58,476.99,36.89,160.31,289.97,469.88,1424.61,26.04,14.29,0.60,70.75,11.99,93.69,RO,N,42780.310784,1357.669117,255.885363,10.663857,7.214261,9.975576,300.040437
1,110002,Ariquemes - RO,2022,96833.0,209,3809355.0,1.193331e+08,1100023,0.702,0.600,0.806,0.716,0.53,689.95,120.08,270.77,420.44,638.10,1996.77,11.54,4.36,0.51,73.36,7.90,98.54,RO,N,39339.429740,1232.359333,215.835511,10.580008,7.117497,11.480753,288.266595
2,110003,Cabixi - RO,2022,5351.0,8,289783.0,1.093209e+07,1100031,0.650,0.559,0.757,0.650,0.51,457.17,78.16,181.72,292.34,480.37,1256.80,21.20,7.27,0.44,70.39,13.63,95.49,RO,N,54154.924313,2042.999993,149.504765,10.899623,7.622664,8.585226,154.284260
3,110004,Cacoal - RO,2022,86887.0,181,3195489.0,7.671583e+07,1100049,0.718,0.620,0.821,0.727,0.57,738.06,108.24,265.14,431.80,656.39,2247.76,13.08,5.97,0.61,74.27,8.29,97.96,RO,N,36777.527133,882.937968,208.316549,10.512669,6.784387,11.372375,259.868382
4,110005,Cerejeiras - RO,2022,15890.0,29,903099.0,2.560032e+07,1100056,0.692,0.602,0.799,0.688,0.50,577.18,104.82,236.32,389.57,581.95,1568.87,13.70,4.72,0.46,72.94,10.29,97.53,RO,N,56834.424166,1611.096328,182.504720,10.947915,7.385291,9.673508,212.808059


---

**Próximo passo:** `03_regressao.ipynb` — modelo de regressão multivariada (XGBoost + linear) sobre `feature_matrix.parquet`, com projeção contrafactual e análise de feature importance.